> If GroupBy summarizes data,   
> **Merge/Join is how real datasets are actually built.**

APIs, databases, logs, ML features — **everything gets merged.**

### Intro
**Three questions** before merging:
1. **What is the join key?**
2. **What is the relationship?**
    - one-to-one
    - one-to-many
    - many-to-many
3. **What should happen to unmatched rows?**

In [1]:
import pandas as pd

users = pd.DataFrame({
    "user_id": [1, 2, 3, 4],
    "name": ["Asha", "Ravi", "Neha", "Karan"],
    "city": ["Pune", "Mumbai", "Delhi", "Pune"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "user_id": [1, 2, 2, 5, 3],
    "amount": [500, 700, 300, 400, 600]
})

### `merge()` - SQL-style Joins

#### Inner Join
- Keeps only matching users
- SQL: `INNER JOIN`

In [2]:
pd.merge(users, orders, on="user_id")

,user_id,name,city,order_id,amount
0,1,Asha,Pune,101,500
1,2,Ravi,Mumbai,102,700
2,2,Ravi,Mumbai,103,300
3,3,Neha,Delhi,105,600


#### Left Join
- Keeps all users
- Missing orders -> `NaN`

In [3]:
pd.merge(users, orders, on="user_id", how="left")

,user_id,name,city,order_id,amount
0,1,Asha,Pune,101.0,500.0
1,2,Ravi,Mumbai,102.0,700.0
2,2,Ravi,Mumbai,103.0,300.0
3,3,Neha,Delhi,105.0,600.0
4,4,Karan,Pune,NaN,NaN


#### Right Join
- Keeps all orders
- Users may be missing

In [4]:
pd.merge(users, orders, on="user_id", how="right")

,user_id,name,city,order_id,amount
0,1,Asha,Pune,101,500
1,2,Ravi,Mumbai,102,700
2,2,Ravi,Mumbai,103,300
3,5,NaN,NaN,104,400
4,3,Neha,Delhi,105,600


#### Outer Join
- Keeps everything
- Missing on either side -> `NaN`

In [5]:
pd.merge(users, orders, on="user_id", how="outer")

,user_id,name,city,order_id,amount
0,1,Asha,Pune,101.0,500.0
1,2,Ravi,Mumbai,102.0,700.0
2,2,Ravi,Mumbai,103.0,300.0
3,3,Neha,Delhi,105.0,600.0
4,4,Karan,Pune,NaN,NaN
5,5,NaN,NaN,104.0,400.0


### Non-Matching Column Names

In [6]:
pd.merge(users, orders, left_on="user_id", right_on="user_id")

,user_id,name,city,order_id,amount
0,1,Asha,Pune,101,500
1,2,Ravi,Mumbai,102,700
2,2,Ravi,Mumbai,103,300
3,3,Neha,Delhi,105,600


The **column names are actually same here** `(user_id)`, so this will be equivalent to:
```python
pd.merge(users, orders, on="user_id")
```

**If key differ:**
```python
pd.merge(users, orders, left_on="user_id", right_on="customer_id")
```

*Assuming orders DataFrame has `customer_id` column*

### Detecting Join Problems
#### Check merge indicator

In [7]:
pd.merge(users, orders, on="user_id", how="left", indicator=True)

,user_id,name,city,order_id,amount,_merge
0,1,Asha,Pune,101.0,500.0,both
1,2,Ravi,Mumbai,102.0,700.0,both
2,2,Ravi,Mumbai,103.0,300.0,both
3,3,Neha,Delhi,105.0,600.0,both
4,4,Karan,Pune,NaN,NaN,left_only


Output column `_merge`:
- `both`
- `left_only`
- `right_only`

### `join()` - Index-Based Joins
Use when:
-  Join key is the index
- Simpler syntax

In [8]:
users.set_index("user_id").join(
    orders.set_index("user_id"),
    how="left"
)

,name,city,order_id,amount
user_id,,,,
1,Asha,Pune,101.0,500.0
2,Ravi,Mumbai,102.0,700.0
2,Ravi,Mumbai,103.0,300.0
3,Neha,Delhi,105.0,600.0
4,Karan,Pune,NaN,NaN


### `concat()` - Stacking Data
#### Row-wise concatenation - `stacks down`
```python
pd.concat([df1, df2], axis=0)
```

#### Column-wise concatenation - `stacks side by side`
```python
pd.concat([df1, df2], axis=1)
```


### Handling Duplicate Column Names

In [9]:
users = pd.DataFrame({
    "user_id": [1, 2],
    "status": ["active", "inactive"],
    "created_at": ["2023-01-01", "2023-01-05"]
})

orders = pd.DataFrame({
    "order_id": [101, 102],
    "user_id": [1, 2],
    "status": ["shipped", "pending"],
    "created_at": ["2023-02-10", "2023-02-12"]
})

Both DataFrames share:
- `user_id` (join key)
- `status`
- `created_at`

#### Merge with suffixes

In [10]:
pd.merge(users, orders, on="user_id", suffixes=("_user", "_order"))

,user_id,status_user,created_at_user,order_id,status_order,created_at_order
0,1,active,2023-01-01,101,shipped,2023-02-10
1,2,inactive,2023-01-05,102,pending,2023-02-12


### Relationship

#### One-to-One (1:1)
Meaning:
- Each key appears **once** in both tables
- One row matches **exactly one row**

In [11]:
users = pd.DataFrame({
    "user_id": [1, 2],
    "name": ["Alice", "Bob"]
})

profiles = pd.DataFrame({
    "user_id": [1, 2],
    "age": [25, 32]
})

In [12]:
pd.merge(users, profiles, on="user_id")

,user_id,name,age
0,1,Alice,25
1,2,Bob,32


Key point
- Row count stays the same
- No duplication

#### One-to-Many (1:N)
Meaning:
- One row in the **left table** matches **multiple rows** in the right
- Very common in databases

In [13]:
users = pd.DataFrame({
    "user_id": [1, 2],
    "name": ["Alice", "Bob"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103],
    "user_id": [1, 1, 2]
})

In [14]:
pd.merge(users, orders, on="user_id")

,user_id,name,order_id
0,1,Alice,101
1,1,Alice,102
2,2,Bob,103


Key point
- **Left row is duplicated**
- One match -> many output rows

#### Many-to-Many (M:N)
Meaning
- Multiple rows on **both sides** share the same key
- Creates a **cartesian effect**

In [15]:
students = pd.DataFrame({
    "course": ["Math", "Math", "Science"],
    "student": ["Alice", "Bob", "Alice"]
})

grades = pd.DataFrame({
    "course": ["Math", "Math", "Science"],
    "grade": ["A", "B", "A+"]
})


In [16]:
pd.merge(students, grades, on="course")

,course,student,grade
0,Math,Alice,A
1,Math,Alice,B
2,Math,Bob,A
3,Math,Bob,B
4,Science,Alice,A+


Key point
- **Row explosion**
- Every combination is produced

| Relationship | Key uniqueness     | Row behavior   |
| ------------ | ------------------ | -------------- |
| One-to-One   | Unique in both     | No duplication |
| One-to-Many  | Unique on one side | Rows repeat    |
| Many-to-Many | Duplicates on both | Row explosion  |


#### Validate relationships
This raises an error if the relationship is violated.

In [17]:
pd.merge(
    users,
    orders,
    on="user_id",
    validate="one_to_many"
)

,user_id,name,order_id
0,1,Alice,101
1,1,Alice,102
2,2,Bob,103


### Production Checklist for Joins
#### Before merging:
```python
df["key"].is_unique
```

#### After merging:
```python
assert len(result) >= max(len(left), len(right))
```

#### Validate:
```python
validate="one_to_many"
```

####vInspect:
```python
indicator=True
```

### Base DataFrames

In [18]:
users = pd.DataFrame({
    "user_id": [1, 2, 3, 4],
    "name": ["Asha", "Ravi", "Neha", "Karan"],
    "city": ["Pune", "Mumbai", "Delhi", "Pune"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106],
    "user_id": [1, 2, 2, 5, 3, 3],
    "amount": [500, 700, 300, 400, 600, 200]
})

payments = pd.DataFrame({
    "order_id": [101, 102, 103, 105],
    "payment_mode": ["UPI", "Card", "UPI", "Cash"]
})

### Exercise 1
Answer in words:
1. What is the relationship between:
   - `users -> orders`
   - `orders -> payments`
2. Which table should be the **left table** in:
   - an API returning all users and their orders?
   - an analytics report of total orders?

1. Relationship:
    - user -> orders: One to Many
    - orders -> payments: One to Zero or One
2. For left join:
    - users
    - orders

### Exercise 2
1. Merge `users` and `orders` so that:
   - **all users appear**
   - orders appear if they exist
2. Use the correct join type
3. Inspect:
   - number of rows before & after merge

In [19]:
pd.merge(users, orders, on="user_id", how="left")

,user_id,name,city,order_id,amount
0,1,Asha,Pune,101.0,500.0
1,2,Ravi,Mumbai,102.0,700.0
2,2,Ravi,Mumbai,103.0,300.0
3,3,Neha,Delhi,105.0,600.0
4,3,Neha,Delhi,106.0,200.0
5,4,Karan,Pune,NaN,NaN


Row count increases - expected

### Exercise 3
1. Re-run the merge using:
```python
indicator=True
```
2. Answer:
   - Which users have no orders?
   - Which orders have no users?

In [20]:
pd.merge(users, orders, on="user_id", how="left", indicator=True)

,user_id,name,city,order_id,amount,_merge
0,1,Asha,Pune,101.0,500.0,both
1,2,Ravi,Mumbai,102.0,700.0,both
2,2,Ravi,Mumbai,103.0,300.0,both
3,3,Neha,Delhi,105.0,600.0,both
4,3,Neha,Delhi,106.0,200.0,both
5,4,Karan,Pune,NaN,NaN,left_only


- Users with `_merge == "left_only"` -> **no orders**
- Orders with `user_id = 5` -> **no matching user**

### Exercise 4
1. Merge users + orders again
2. Use `validate=` to enforce the relationship
3. Try:
   - `one_to_one`
   - `one_to_many`
4. Observe:
   - Which passes?
   - Which fails?
   - Why?

In [22]:
pd.merge(users, orders, on="user_id", how="left", validate="one_to_many")

,user_id,name,city,order_id,amount
0,1,Asha,Pune,101.0,500.0
1,2,Ravi,Mumbai,102.0,700.0
2,2,Ravi,Mumbai,103.0,300.0
3,3,Neha,Delhi,105.0,600.0
4,3,Neha,Delhi,106.0,200.0
5,4,Karan,Pune,NaN,NaN


`validate="one-to-one"` will fail

### Exercise 5
1. Starting from users + orders:
   - join payments
2. Ensure:
   - missing payments appear as `NaN`
3. Count:
   - how many orders have no payment

In [25]:
users_orders_payments = users.merge(orders, on="user_id", how="left").merge(payments, on="order_id", how="left")
users_orders_payments

,user_id,name,city,order_id,amount,payment_mode
0,1,Asha,Pune,101.0,500.0,UPI
1,2,Ravi,Mumbai,102.0,700.0,Card
2,2,Ravi,Mumbai,103.0,300.0,UPI
3,3,Neha,Delhi,105.0,600.0,Cash
4,3,Neha,Delhi,106.0,200.0,NaN
5,4,Karan,Pune,NaN,NaN,NaN


In [26]:
users_orders_payments["payment_mode"].isna().sum()

np.int64(2)

### Exercise 6
1. Aggregate orders to:
   - total order amount per user
   - number of orders per user
2. Merge this aggregated result back into `users`
3. Ensure:
   - row count == number of users

In [30]:
order_agg = (
    orders
    .groupby("user_id")
    .agg(
        total_amount=("amount", "sum"),
        order_count=("order_id", "count")
    )
    .reset_index()
)

order_agg

,user_id,total_amount,order_count
0,1,500,1
1,2,1000,2
2,3,800,2
3,5,400,1


- **Split** orders by user
- **Apply** aggregations
- **Combine** into one row per user

User 2 -> 2 orders -> 1 row   
User 3 -> 2 orders -> 1 row

In [31]:
users_features = pd.merge(
    users,
    order_agg,
    on="user_id",
    how="left"
)

users_features

,user_id,name,city,total_amount,order_count
0,1,Asha,Pune,500.0,1.0
1,2,Ravi,Mumbai,1000.0,2.0
2,3,Neha,Delhi,800.0,2.0
3,4,Karan,Pune,NaN,NaN


In [29]:
len(users_features) == len(users)  # True

True

### Exercise 7
1. Set `user_id` as index in both `users` and aggregated orders
2. Use `.join()` instead of `merge`
3. Confirm result matches Exercise 6

In [ ]:
users_idx = users.set_index("user_id")
order_agg_idx = order_agg.set_index("user_id")

users_joined = users_idx.join(order_agg_idx, how="left").reset_index()

### Exercise 8
#### Row-wise concat
1. Create a new DataFrame:
```python
new_users = pd.DataFrame({
    "user_id": [5, 6],
    "name": ["Meena", "Suresh"],
    "city": ["Chennai", "Hyderabad"]
})
```
2. Append it to `users` using `concat`
3. Reset index properly

#### Column-wise concat
1. Create:
```python
user_flags = pd.DataFrame({
    "is_active": [True, True, False, True]
})
```
2. Concatenate this column-wise with `users`
3. Observe what happens if index is misaligned


In [33]:
new_users = pd.DataFrame({
    "user_id": [5, 6],
    "name": ["Meena", "Suresh"],
    "city": ["Chennai", "Hyderabad"]
})

users_all = pd.concat([users, new_users], ignore_index=True)
users_all

,user_id,name,city
0,1,Asha,Pune
1,2,Ravi,Mumbai
2,3,Neha,Delhi
3,4,Karan,Pune
4,5,Meena,Chennai
5,6,Suresh,Hyderabad


In [34]:
user_flags = pd.DataFrame({
    "is_active": [True, True, False, True]
})

users_with_flags = pd.concat(
    [users, user_flags],
    axis=1
)

users_with_flags

,user_id,name,city,is_active
0,1,Asha,Pune,True
1,2,Ravi,Mumbai,True
2,3,Neha,Delhi,False
3,4,Karan,Pune,True
